In [0]:
# ============================================
# TABLA GOLD 1: Análisis por ciudad
# ============================================

print("\n" + "="*70)
print("CREACIÓN DE TABLA GOLD - CIUDADES")
print("="*70)

df_gold_city = spark.sql("""
SELECT 
    town_city,
    COUNT(1) as total_transactions,
    ROUND(AVG(price), 0) as avg_price,
    ROUND(MIN(price), 0) as min_price,
    ROUND(MAX(price), 0) as max_price,
    PERCENTILE_APPROX(price, 0.5) as median_price
FROM workspace.uk_housing.silver_property_sales
WHERE town_city != 'Unknown'
GROUP BY town_city
HAVING total_transactions >= 1000
""")

# Guardar con overwriteSchema
df_gold_city.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.uk_housing.gold_city_prices")

print("\n Tabla Gold creada: gold_city_prices")
print(f" Total de ciudades: {df_gold_city.count():,}")
print("\n--- Top 10 ciudades con más transacciones ---")
display(df_gold_city.limit(10))

In [0]:
# ============================================
# TABLA GOLD 2: Evolución temporal
# ============================================

print("\n" + "="*70)
print("CREACIÓN DE TABLA GOLD - EVOLUCIÓN TEMPORAL")
print("="*70)

df_gold_temporal = spark.sql("""
SELECT 
    year,
    month,
    town_city,
    COUNT(*) as total_transactions,
    ROUND(AVG(price), 0) as avg_price,
    ROUND(STDDEV(price), 0) as std_price,
    SUM(price) AS total_value
    
FROM workspace.uk_housing.silver_property_sales
GROUP BY year, month, town_city
ORDER BY year, month;
""")

# Guardar
df_gold_temporal.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.uk_housing.gold_temporal_trends")

print("\n Tabla Gold creada: gold_temporal_trends")
print("\nÚltimos 12 meses:")
display(df_gold_temporal.orderBy("year", "month", ascending=False).limit(12))

### Análisis por ciudad

In [0]:
# ============================================
# TABLA GOLD 3: Análisis por tipo de propiedad y ciudad
# ============================================

print("\n" + "="*70)
print("CREACIÓN DE TABLA GOLD - TIPO DE PROPIEDAD")
print("="*70)

df_gold_property = spark.sql("""
SELECT 
    town_city,
    property_type_desc,
    COUNT(1) as total_transactions,
    ROUND(AVG(price), 0) as avg_price,
    ROUND(MIN(price), 0) as min_price,
    ROUND(MAX(price), 0) as max_price,
    ROUND(SUM(price), 0) as total_price
FROM workspace.uk_housing.silver_property_sales
GROUP BY town_city, property_type_desc
ORDER BY avg_price DESC
""")

# Guardar
df_gold_property.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.uk_housing.gold_property_analysis")

print("\n Tabla Gold creada: gold_property_analysis")
display(df_gold_property.limit(20))
     

In [0]:
# ============================================
# TABLA GOLD 4: Análisis de precios por distrito y evolución temporal
# ============================================

print("\n" + "="*70)
print("CREACIÓN DE TABLA GOLD - ANÁLISIS GEOGRÁFICO POR DISTRITO")
print("="*70)

df_gold_district = spark.sql("""
SELECT 
    town_city,
    district,
    year,
    month,
    COUNT(1) as total_transactions,
    ROUND(AVG(price), 0) as avg_price,
    ROUND(MIN(price), 0) as min_price,
    ROUND(MAX(price), 0) as max_price
FROM workspace.uk_housing.silver_property_sales
GROUP BY town_city, district, year, month
ORDER BY year, month
""")

# Guardar
df_gold_district.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.uk_housing.gold_district_analysis")

print("\n Tabla Gold creada: gold_district_analysis")
display(df_gold_district.limit(20))

In [0]:
# ============================================
# TABLA GOLD 5: Análisis por categorías de precio por ciudad
# ============================================
print("\n" + "="*70)
print("CREACIÓN DE TABLA GOLD - CATEGORÍAS DE PRECIO")
print("="*70)

df_gold_price_category = spark.sql("""
SELECT 
    town_city,
    price_category,
    COUNT(*) as total_transactions,
    ROUND(AVG(price), 0) as avg_price,
    ROUND(MIN(price), 0) as min_price,
    ROUND(MAX(price), 0) as max_price
FROM workspace.uk_housing.silver_property_sales
GROUP BY town_city, price_category
ORDER BY avg_price DESC
""")

# Guardar
df_gold_price_category.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.uk_housing.gold_price_category_analysis")

print("\n Tabla Gold creada: gold_price_category_analysis")
display(df_gold_price_category.limit(20))

In [0]:
# ============================================
# RESUMEN FINAL DEL NOTEBOOK 2
# ============================================

print("\n" + "="*70)
print("RESUMEN FINAL - ARQUITECTURA MEDALLION")
print("="*70)

print("\nTablas creadas:")
spark.sql("SHOW TABLES IN workspace.uk_housing").filter("tableName LIKE '%property%' OR tableName LIKE '%gold%'").show(truncate=False)

print("\nEstadísticas:")
print(f"  Bronze (raw): {spark.table('workspace.uk_housing.bronze_property_sales').count():,} registros")
print(f"  Silver (clean): {spark.table('workspace.uk_housing.silver_property_sales').count():,} registros")
print(f"  Gold - Ciudades: {spark.table('workspace.uk_housing.gold_city_prices').count():,} ciudades")
print(f"  Gold - Temporal: {spark.table('workspace.uk_housing.gold_temporal_trends').count():,} periodos")
print(f"  Gold - Propiedades: {spark.table('workspace.uk_housing.gold_property_analysis').count():,} combinaciones")
print(f"  Gold - Distritos: {spark.table('workspace.uk_housing.gold_district_analysis').count():,} combinaciones")
print(f"  Gold - Categorías de precios: {spark.table('workspace.uk_housing.gold_price_category_analysis').count():,} combinaciones")

print("\n Notebook 2 completado exitosamente")